# 검색어 연결표 만들기

검색어 표(`search_terms`가 만든 `source/search/queries_all.csv`)의 줄마다 **그 검색어가 종목과 그래프에서 어떻게 이어져 있는지**(관계 종류·내용·근거 문장)를 붙여 `source/search/term_links.csv`로 내보내고, 서버 DB `news.term_links`에 올린다(`graphDB.md` 6.6).

- 쓰는 곳: 뉴스 상세의 "예상과 밤사이 소식, 같이 읽기"에서 **이 기사가 어떤 경로로 내 종목과 이어졌는지** 설명하는 문장을 LLM이 쓸 때의 재료. 서버에는 그래프 DB가 없으므로, 기사와 상관없이 정해진 "종목–검색어" 관계만 표로 올려 둔다.
- 기사와 이 표는 수집 DB의 `hits.term`(검색 문자열)과 이 표의 `query`로 잇는다. 같은 검색 문자열이 여러 종목에 있으면 기사도 그 종목들 모두와 이어진다.
- 검색어 표를 다시 만들면(그래프를 다시 만들 때) 이 노트북도 다시 돌린다. 서버 표는 통째로 바꾼다.

연결 방식(`link`):

| link | 검색어 종류 | 그래프에서 찾는 것 |
|---|---|---|
| direct | 구체 제품·넓은 제품·원자재·지표·브랜드·해외 상대·테마 대표·정책 | 종목 노드와 검색어 노드 사이의 관계(양방향, 모든 관계 파일) |
| self | 자기 회사·회사 기본 | 검색어 노드가 종목 자신 |
| sector | 업종 기본 | 종목의 소속 업종(`IN_SECTOR`, WI26 업종 이름) |
| theme | 테마 기본 | 종목의 소속 테마(`HAS_THEME`) |
| theme_product | 직접 관계가 없는 테마 대표 | 그 테마 검색어의 테마에 종목이 속함(`HAS_THEME`) |
| none | 위 어디에도 안 이어짐 | — |

## 1. 설정

In [ ]:
import json, os
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()                                   # graph_db
PROJ = ROOT.parent
SEARCH = ROOT / 'source' / 'search'
NEO = ROOT / 'source' / 'graph' / 'neo4j'           # graph_db.ipynb가 만든 노드·관계 표
QUERIES_PATH = SEARCH / 'queries_all.csv'           # search_terms가 만든 검색어 표(951종목)
OUT_PATH = SEARCH / 'term_links.csv'                # 이 노트북의 결과(데이터라 git에는 없음)
MAX_EVIDENCE = 2                                    # 관계마다 남길 근거 문장 수
EVIDENCE_CHARS = 400                                # 근거 문장 한 개의 최대 글자 수(DART 표 줄이 길어서)

DB_ENV = 'GNEWS_DB_URL'                             # 프로젝트 루트 .env의 서버 DB 주소(수집 계정, news 영역 쓰기)
DB_TABLE = 'term_links'                             # news.term_links
RUN_UPLOAD = False                                  # True여야 서버 DB에 올린다(표를 통째로 바꿈)

# 6절 검색어 한글 이름(화면 칸에 적을 이름, 2026-11-21 더함)
NAMES_PATH = SEARCH / 'term_names_ko.csv'           # 번역해 둔 이름(검색어마다 한 줄). 있는 검색어는 다시 번역하지 않는다
NAME_PROMPT_PATH = ROOT / 'prompts' / 'term_name_ko_v1.md'
LLM_MODEL = 'gpt-6.1-terra'
RUN_LLM = False                                     # True여야 OpenAI로 번역한다(아래 NAME_LIMIT만큼)
NAME_LIMIT = 20                                     # 이번에 번역할 최대 검색어 수(None이면 남은 것 전부). 처음엔 20개로 시험
NAME_BATCH = 200                                    # 한 번 호출에 넣을 검색어 수
RUN_NAME_UPLOAD = False                             # True면 서버 표의 한글 이름 칸(name_ko)만 채운다(표 전체를 바꾸지 않음)

# 관계 파일 이름 → 화면·LLM에 줄 한국어 이름
REL_LABEL = {
    'SUPPLIES': '공급', 'BUYS': '구매', 'WINS_ORDER': '수주', 'PRODUCES': '생산', 'PARTNERS': '협력', 'COMPETES': '경쟁',
    'HAS_BRAND': '브랜드', 'OWNS': '소유', 'INVESTS': '투자', 'LICENSES': '기술 계약', 'BUILDS_OPERATES': '건설·운영',
    'DISPUTES': '분쟁', 'DIVESTS': '매각', 'FUNDS': '자금 지원', 'AFFECTS': '영향', 'POLICY_STANCE': '정책 입장',
    'REGULATES': '규제', 'OF_PRODUCT': '제품', 'SUBTYPE_OF': '하위 종류', 'IN_GROUP': '같은 그룹', 'OTHER': '기타',
    'IN_SECTOR': '소속 업종', 'HAS_THEME': '소속 테마',
}
print(QUERIES_PATH.exists(), NEO.exists())

## 2. 읽기

검색어 표, 노드 이름(한국어 이름이 있으면 그것), 관계 표 전부. 관계 표의 목록 칸(`kinds`·`notes`·`evidence` 등)은 JSON 문자열이라 풀어 둔다.

In [ ]:
queries = pd.read_csv(QUERIES_PATH, dtype=str, encoding='utf-8-sig').fillna('')
print('검색어', len(queries), '줄 · 종목', queries.stock_code.nunique())

def as_list(v):
    """관계 표의 JSON 목록 칸 → 파이썬 목록(비었거나 깨졌으면 빈 목록)"""
    if not isinstance(v, str) or not v.strip():
        return []
    try:
        x = json.loads(v)
        return x if isinstance(x, list) else [x]
    except ValueError:
        return [v]

# 노드 이름: id → 보여 줄 이름
node_name = {}
for f in sorted(NEO.glob('nodes_*.csv')):
    d = pd.read_csv(f, dtype=str, encoding='utf-8-sig').fillna('')
    for r in d.itertuples(index=False):
        r = r._asdict()
        node_name[r['id']] = r.get('name_ko') or r.get('name') or r['id']
sector_id = {n: i for i, n in node_name.items() if i.startswith('WI:')}       # 업종 이름 → WI 노드
theme_id = {i[3:]: i for i in node_name if i.startswith('TH:')}               # 테마 이름 → TH 노드

# 관계: (start, end) → 관계 목록
rels = {}
for f in sorted(NEO.glob('rels_*.csv')):
    typ = f.stem[5:]
    d = pd.read_csv(f, dtype=str, encoding='utf-8-sig').fillna('')
    for r in d.to_dict('records'):
        rels.setdefault((r['start'], r['end']), []).append(dict(
            type=typ, label=REL_LABEL.get(typ, typ), kinds=as_list(r.get('kinds')), notes=as_list(r.get('notes')),
            amounts=as_list(r.get('amounts')), sources=as_list(r.get('sources')),
            evidence=[str(e)[:EVIDENCE_CHARS] for e in as_list(r.get('evidence'))[:MAX_EVIDENCE]],
            refs=as_list(r.get('evidence_refs'))[:MAX_EVIDENCE]))
print('노드 이름', len(node_name), '· 관계 짝', len(rels), '· 업종', len(sector_id), '· 테마', len(theme_id))

## 3. 검색어마다 종목과의 연결 찾기

관계 방향(`dir`)은 종목 기준이다. `종목→상대`는 "종목이 상대에게 공급한다"처럼 종목이 주어, `상대→종목`은 상대가 주어.

In [ ]:
def directed(a, b, d):
    """(a, b) 관계 목록에 방향 표시를 붙여 돌려준다"""
    return [dict(r, dir=d) for r in rels.get((a, b), [])]

def link_of(row):
    stock, node, kind = 'KR:' + row['stock_code'], row['node'], row['kind']
    if node == stock:
        return 'self', []
    if node.startswith('S:'):                       # 업종 기본: 검색어 표는 'S:업종 이름', 그래프는 'WI:코드'
        wi = sector_id.get(node[2:])
        return ('sector', directed(stock, wi, '종목→상대')) if wi and (stock, wi) in rels else ('none', [])
    if node.startswith('T:'):                       # 테마 기본: 'T:테마' ↔ 'TH:테마'
        th = theme_id.get(node[2:])
        return ('theme', directed(stock, th, '종목→상대')) if th and (stock, th) in rels else ('none', [])
    found = directed(stock, node, '종목→상대') + directed(node, stock, '상대→종목')
    if found:
        return 'direct', found
    th = theme_id.get(row['theme']) if row['theme'] else None
    if th and (stock, th) in rels:                  # 직접 관계는 없고 그 테마에 속함(테마 대표 제품)
        return 'theme_product', directed(stock, th, '종목→상대')
    return 'none', []

def summary(link, found, row):
    """사람이 읽는 한 줄: '공급(종목→상대) 감속기 부품 · 8,200억원'"""
    if link == 'self':
        return '자기 회사' if row['kind'] == '자기 회사' else '회사 기본 사업'
    if link == 'sector':
        return f"소속 업종: {row['node'][2:]}"
    if link == 'theme':
        return f"소속 테마: {row['node'][2:]}"
    if link == 'theme_product':
        return f"소속 테마: {row['theme']} (그 테마의 대표 제품·주제)"
    out = []
    for r in found:
        bits = [f"{r['label']}({r['dir']})"] + r['notes'][:2] + r['amounts'][:1]
        out.append(' '.join(bits[:1]) + (' ' + ' · '.join(bits[1:]) if bits[1:] else ''))
    return ' / '.join(out)

rows = []
for row in queries.to_dict('records'):
    link, found = link_of(row)
    rows.append(dict(
        stock_code=row['stock_code'], company=row['company'], kind=row['kind'], term=row['term'], query=row['query'],
        node=row['node'], node_name=node_name.get(row['node'], row['node'].split(':', 1)[-1]), theme=row['theme'],
        link=link, n_relations=len(found), has_evidence=any(r['evidence'] for r in found),
        summary=summary(link, found, row), relations=json.dumps(found, ensure_ascii=False)))
links = pd.DataFrame(rows)
print(len(links), '줄')

## 4. 결과 보기

In [ ]:
# 검색어 종류 × 연결 방식
print(pd.crosstab(links.kind, links.link, margins=True))
print()
print('근거 문장이 있는 줄', int(links.has_evidence.sum()), '/', len(links))

In [ ]:
# 예: POSCO홀딩스 ← General Motors, 그리고 종류별 한 줄씩
ex = links[(links.company == 'POSCO홀딩스') & (links.term == 'General Motors')]
print(ex[['company', 'kind', 'term', 'node_name', 'link', 'summary']].to_string(index=False))
print(json.dumps(json.loads(ex.relations.iloc[0]), ensure_ascii=False, indent=1)[:1500] if len(ex) else '')
print()
print(links.groupby('kind').head(1)[['company', 'kind', 'term', 'link', 'summary']].to_string(index=False))

## 5. 파일로 저장

In [ ]:
links.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print(OUT_PATH, len(links), '줄')

## 6. 검색어 한글 이름(화면용)

연결 그림의 칸에 영어 검색어 대신 적을 짧은 한글 이름이다(예: "hot-rolled steel sheet and coil" → 열연강판, "General Motors" → GM). 2026-11-21 정함.

- 자기 회사 검색어는 번역하지 않고 종목 이름을 쓴다(회사 기본은 'food prices'처럼 사업 낱말이라 번역한다).
- 나머지는 검색어(따옴표를 뗀 것)마다 한 번만 번역해 `term_names_ko.csv`에 쌓는다. 뜻을 고르도록 종류와 그 검색어를 쓰는 종목 하나를 같이 넘긴다. 이미 있는 검색어는 건너뛰므로, 검색어 표를 다시 만들어도 새 검색어만 번역한다.
- 지시문: `prompts/term_name_ko_v1.md`. OpenAI 키는 `.env`의 `OPENAI_API_KEY`이고 값은 출력하지 않는다.
- 비용(2026-11-21): 2,703개, 입력 약 9만·출력 약 5~10만 토큰으로 일반 호출 약 $0.4~0.6(넉넉히 $1).

In [ ]:
import datetime as dt, urllib.request, urllib.error

def env_value(name):   # .env 값(키 값은 출력하지 않는다. 7절 DB 주소에도 쓴다)
    if os.environ.get(name):
        return os.environ[name]
    for line in (PROJ / '.env').read_text(encoding='utf-8').splitlines():
        if line.startswith(name + '='):
            return line.split('=', 1)[1].strip().strip('"').strip("'")
    return None


def plain(q):
    # 검색 문자열의 따옴표를 모두 뗀다('"Nvidia" "memory chips"' → 'Nvidia memory chips')
    return ' '.join(str(q).replace('"', ' ').split())

def load_names():
    return pd.read_csv(NAMES_PATH, dtype=str).fillna('') if NAMES_PATH.exists() else pd.DataFrame(columns=['q', 'ko', 'kind', 'model', 'created_kst'])

def ask_names(items):
    """검색어 묶음 → {q: 한글 이름}. 오류는 종류·코드만 알린다."""
    schema = {'type': 'object', 'additionalProperties': False, 'required': ['names'], 'properties': {'names': {'type': 'array', 'items': {
        'type': 'object', 'additionalProperties': False, 'required': ['q', 'ko'], 'properties': {'q': {'type': 'string'}, 'ko': {'type': 'string'}}}}}}
    body = {'model': LLM_MODEL, 'instructions': NAME_PROMPT_PATH.read_text(encoding='utf-8'), 'store': False, 'reasoning': {'effort': 'low'},
            'input': 'terms json:\n' + json.dumps({'terms': items}, ensure_ascii=False),
            'text': {'format': {'type': 'json_schema', 'name': 'term_names', 'strict': True, 'schema': schema}}}
    req = urllib.request.Request('https://api.openai.com/v1/responses', method='POST', data=json.dumps(body).encode('utf-8'),
                                 headers={'Authorization': f"Bearer {env_value('OPENAI_API_KEY')}", 'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=600) as r:
            res = json.loads(r.read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        raise RuntimeError(f'OpenAI HTTP {e.code}') from None
    text = ''.join(c.get('text', '') for o in res.get('output', []) if o.get('type') == 'message' for c in o.get('content', []))
    return {x['q']: x['ko'].strip() for x in json.loads(text)['names']}, res.get('usage') or {}

def translate_names():
    names = load_names()
    done = set(names.q)
    todo = (links[links.kind != '자기 회사'].assign(q=lambda d: d['query'].map(plain))
            .drop_duplicates('q')[['q', 'kind', 'company']])
    todo = todo[~todo.q.isin(done)]
    print('번역할 검색어', len(todo), '개 (이미 있음', len(done), '개)')
    if NAME_LIMIT is not None:
        todo = todo.head(NAME_LIMIT)
    if not RUN_LLM or todo.empty:
        print('RUN_LLM = False라 OpenAI를 부르지 않음' if not RUN_LLM else '새로 번역할 검색어 없음')
        return names
    tin = tout = 0
    for i in range(0, len(todo), NAME_BATCH):
        part = todo.iloc[i:i + NAME_BATCH]
        got, usage = ask_names(part.to_dict('records'))
        tin += usage.get('input_tokens', 0); tout += usage.get('output_tokens', 0)
        miss = [q for q in part.q if not got.get(q)]
        if miss:
            print('주의: 이름이 빠진 검색어(다음에 다시):', miss[:10])
        now = dt.datetime.now(dt.timezone(dt.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')
        new = pd.DataFrame([{'q': q, 'ko': got[q], 'kind': k, 'model': LLM_MODEL, 'created_kst': now} for q, k in zip(part.q, part.kind) if got.get(q)])
        new.to_csv(NAMES_PATH, mode='a', header=not NAMES_PATH.exists(), index=False, encoding='utf-8-sig')   # 묶음마다 바로 저장
        print(f'{i + len(part)}/{len(todo)} 저장 · 토큰 입력 {tin:,} 출력 {tout:,} · 약 ${(tin * 1 + tout * 5) / 1e6:.2f}')
    return load_names()

names = translate_names()
ko = dict(zip(names.q, names.ko))
links['name_ko'] = [r.company if r.kind == '자기 회사' else ko.get(plain(r.query), '') for r in links.itertuples()]
print('한글 이름 있는 줄', int((links.name_ko != '').sum()), '/', len(links))
print(links[links.name_ko != ''].drop_duplicates('query').groupby('kind').head(2)[['company', 'kind', 'query', 'name_ko']].to_string(index=False))

## 7. 서버 DB에 올리기

`news.term_links`를 통째로 바꾼다(한 묶음 안에서 지우고 다시 넣어서, 중간에 실패하면 이전 표가 그대로 남음). 키는 (종목코드, 종류, 검색 문자열, 노드). 화면·처리 계정(`gnews_read`·`gnews_app`)은 읽기만 한다.

In [ ]:
DDL = f"""
create table if not exists news.{DB_TABLE} (
  stock_code text not null, company text, kind text not null, term text, query text not null, node text not null,
  node_name text, theme text, link text, n_relations integer, has_evidence boolean, summary text, relations jsonb, name_ko text,
  built_at timestamptz default now(),
  primary key (stock_code, kind, query, node))"""

if RUN_UPLOAD:
    import sqlalchemy as sa
    engine = sa.create_engine(env_value(DB_ENV), pool_pre_ping=True)
    cols = ['stock_code', 'company', 'kind', 'term', 'query', 'node', 'node_name', 'theme', 'link', 'n_relations', 'has_evidence', 'summary', 'relations', 'name_ko']
    data = [{c: (bool(r[c]) if c == 'has_evidence' else int(r[c]) if c == 'n_relations' else r[c]) for c in cols} for r in links.to_dict('records')]
    vals = ', '.join(f'cast(:{c} as jsonb)' if c == 'relations' else ':' + c for c in cols)   # relations는 JSON 문자열 → jsonb
    ins = sa.text(f"insert into news.{DB_TABLE} ({', '.join(cols)}) values ({vals})")
    with engine.begin() as con:
        con.exec_driver_sql(DDL)
        con.exec_driver_sql(f'alter table news.{DB_TABLE} add column if not exists name_ko text')
        con.exec_driver_sql(f'create index if not exists {DB_TABLE}_query on news.{DB_TABLE} (query)')
        con.exec_driver_sql(f'delete from news.{DB_TABLE}')
        for i in range(0, len(data), 2000):
            con.execute(ins, data[i:i + 2000])
        con.exec_driver_sql(f'grant select on news.{DB_TABLE} to gnews_read, gnews_app')
        n = con.exec_driver_sql(f'select count(*), count(*) filter (where has_evidence) from news.{DB_TABLE}').one()
    print('서버 DB news.term_links:', n[0], '줄 (근거 있음', n[1], ')')
elif RUN_NAME_UPLOAD:
    # 표는 그대로 두고 한글 이름 칸만 채운다(검색 문자열이 같은 줄에 같은 이름)
    import sqlalchemy as sa
    engine = sa.create_engine(env_value(DB_ENV), pool_pre_ping=True)
    pairs = links[['query', 'kind', 'stock_code', 'name_ko']].drop_duplicates().to_dict('records')
    with engine.begin() as con:
        con.exec_driver_sql(f'alter table news.{DB_TABLE} add column if not exists name_ko text')
        upd = sa.text(f'update news.{DB_TABLE} set name_ko = :name_ko where query = :query and kind = :kind and stock_code = :stock_code')
        for i in range(0, len(pairs), 2000):
            con.execute(upd, pairs[i:i + 2000])
        n = con.exec_driver_sql(f"select count(*), count(*) filter (where coalesce(name_ko, '') <> '') from news.{DB_TABLE}").one()
    print('서버 DB news.term_links 한글 이름:', n[1], '/', n[0], '줄')
else:
    print('RUN_UPLOAD·RUN_NAME_UPLOAD = False라 올리지 않음')